# Synthetic Image Source Attribution

This notebook solves a 10-class image attribution problem: given a synthetic face image, identify which of 10 text-to-image models generated it. The dataset comes from the ICANN 2026 DLMMDD Workshop Challenge.

We use a dual-stream architecture that processes each image twice once as RGB and once through SRM noise filters then fuses both feature sets for classification. By the end of this notebook we'll have a trained model and a `submission.csv` ready to upload.

## Setup

Let's start by downloading the competition data and importing everything we need.

In [ ]:
import kagglehub

path = kagglehub.competition_download('dlmmdd-workshop-synthetic-source-attribution-challenge')

print("Path to competition files:", path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import cv2
%matplotlib inline

# PyTorch core
import torch
import torch.nn as nn

# TorchVision
import torchvision.transforms.v2 as v2
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torchvision.utils import make_grid

# Data utilities
from torch.utils.data.dataloader import DataLoader
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## Configuration

All hyperparameters and paths live in one place so they're easy to change.

In [ ]:
MAIN_PATH = "/kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data"
TRAIN_PATH = os.path.join(MAIN_PATH, 'Training')
TEST_PATH  = os.path.join(MAIN_PATH, 'Test')
TRAIN_CSV_PATH     = os.path.join(MAIN_PATH, 'training.csv')
TEST_CSV_PATH      = os.path.join(MAIN_PATH, 'test.csv')
LABEL_TO_NAMES_PATH = os.path.join(MAIN_PATH, 'sources.txt')

NUM_OF_CLASSES = 10

# 384 is EfficientNet-B4's native resolution, giving richer feature maps than 224
SIZE       = 384
BATCH_SIZE = 64
EPOCHS     = 20
PATIENCE   = 5
LR         = 1e-4

# ImageNet statistics used to normalize inputs for pretrained models
MEAN_NORM = [0.485, 0.456, 0.406]
STD_NORM  = [0.229, 0.224, 0.225]

NUM_WORKERS = 4
PIN_MEMORY  = True

## Exploratory Data Analysis

Before building anything, let's get a feel for the data class distribution, image formats, and what the images actually look like.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
train_df

In [ ]:
train_df['y'].value_counts()

In [ ]:
def show_all_extensions(path):
    """
    Collect all unique file extensions found in a directory.

    Args:
        path: directory path to scan

    Returns:
        list of unique extension strings (e.g. ['jpg', 'png'])
    """
    imgs_names = os.listdir(path)
    result = []
    for img_name in imgs_names:
        x = img_name.split('.')[-1]
        if x not in result:
            result.append(x)
    return result

show_all_extensions(TRAIN_PATH)

In [ ]:
def show_all_imgs_sizes(path):
    """
    Collect all unique image shapes found in a directory.

    Args:
        path: directory path to scan

    Returns:
        list of unique shape tuples (H, W, C)
    """
    imgs_names = os.listdir(path)
    all_sizes = []
    for img_name in imgs_names:
        img_shape = plt.imread(os.path.join(path, img_name)).shape
        if img_shape not in all_sizes:
            all_sizes.append(img_shape)
    return all_sizes

show_all_imgs_sizes(TRAIN_PATH)

In [ ]:
def show_img_per_class():
    """Display one random image from each class."""
    unique_classes = sorted(train_df['y'].unique())
    num_classes = len(unique_classes)

    cols = min(4, num_classes)
    rows = (num_classes + cols - 1) // cols

    plt.figure(figsize=(cols * 4, rows * 4))

    for idx, cls in enumerate(unique_classes):
        class_df = train_df[train_df['y'] == cls]
        img_path = np.random.choice(class_df["path"].values)
        img_full_path = os.path.join(TRAIN_PATH, img_path.split('/')[-1])
        img = plt.imread(img_full_path)

        # grayscale images need to be converted before display
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        plt.subplot(rows, cols, idx + 1)
        plt.imshow(img)
        plt.title(f"Class/Label: {cls}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()


show_img_per_class()

## Transforms and Dataset

Now let's define how images get prepared before entering the model.

The training transform includes light augmentation to make the model more robust to the post-processing operations present in the test set. The val/test transform only resizes and normalizes no augmentation.

In [ ]:
def get_transforms(size=SIZE, apply_on_train=False):
    """
    Build a transform pipeline for training or validation/test.

    Args:
        size: target square image size in pixels
        apply_on_train: if True, adds augmentation steps before normalization

    Returns:
        v2.Compose transform pipeline
    """
    base = [
        v2.Resize((size, size)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ]

    augmentation = [
        v2.RandomHorizontalFlip(p=0.5),
        # simulate grayscale conversion, which is one of the test post-processing ops
        v2.RandomGrayscale(p=0.1),
        # keep values small to avoid destroying forensic color signatures
        v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    ]

    # ImageNet mean/std normalization (required for pretrained models)
    tail = [v2.Normalize(MEAN_NORM, STD_NORM)]

    if apply_on_train:
        return v2.Compose(base + augmentation + tail)
    return v2.Compose(base + tail)

In [ ]:
class SyntheticDataset(Dataset):
    """
    PyTorch Dataset for the synthetic image attribution challenge.

    Handles both train/val (returns image + label) and test (returns image + ID)
    via the is_test flag.

    Args:
        dataframe: pandas DataFrame with columns [ID, path, y] for train or [ID, path] for test
        transform: torchvision transform pipeline to apply to each image
        is_test: if True, returns (image, ID) instead of (image, label)
    """
    def __init__(self, dataframe, transform=None, is_test=False):
        self.df = dataframe
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.df.iloc[idx, 1]
        img = Image.open(image_path).convert('RGB')
        if self.transform:
            img = self.transform(img)

        if self.is_test:
            # test set has no labels, return the image ID for submission matching
            id_ = self.df.iloc[idx, 0]
            return img, id_
        else:
            label = self.df.iloc[idx, 2]
            return img, label

## Data Loading

Now we load the CSVs, fix the image paths to point to the correct directory, and split the training set into train and validation.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df  = pd.read_csv(TEST_CSV_PATH)

# replace relative paths in the CSV with absolute paths on disk
train_df['path'] = train_df['path'].apply(lambda x: os.path.join(TRAIN_PATH, x.split('/')[-1]))
test_df['path']  = test_df['path'].apply(lambda x: os.path.join(TEST_PATH,  x.split('/')[-1]))

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
# stratify ensures each class keeps the same proportion in train and val
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["y"]
)

In [ ]:
train_dataset = SyntheticDataset(train_df, get_transforms(SIZE, True))
val_dataset   = SyntheticDataset(val_df,   get_transforms(SIZE))
test_dataset  = SyntheticDataset(test_df,  get_transforms(SIZE), is_test=True)

In [ ]:
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
# shuffle=False for val and test to keep ID order consistent
val_dl   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_dl  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

Let's do a quick sanity check and visualize one batch to confirm the transforms are working correctly.

In [ ]:
def denorm(imgs):
    """
    Reverse ImageNet normalization so images can be displayed correctly.

    Args:
        imgs: normalized tensor of shape (B, C, H, W) or (C, H, W)

    Returns:
        tensor in [0, 1] range with same shape as input
    """
    mean = torch.tensor(MEAN_NORM).view(1, 3, 1, 1).to(imgs.device)
    std  = torch.tensor(STD_NORM).view(1, 3, 1, 1).to(imgs.device)

    if imgs.dim() == 3:
        imgs = imgs.unsqueeze(0)

    return imgs * std + mean

In [ ]:
def show_grid_images(dataloader):
    """Display a grid of images from one batch (useful for sanity-checking augmentation)."""
    imgs, labels = next(iter(dataloader))
    plt.figure(figsize=(30, 30))
    plt.imshow(make_grid(denorm(imgs), nrow=16).permute(1, 2, 0))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

show_grid_images(train_dl)

## Model Architecture

We use two components before building the full model.

**SRMConv** applies Spatial Rich Model filters fixed convolutional kernels from the image forensics literature. They extract high-frequency noise residuals that reveal traces left by each generative model, patterns invisible to the naked eye.

**DualStreamModel** runs two parallel EfficientNet-B4 backbones: one on the raw RGB image, one on the SRM-filtered version. The features from both streams are concatenated and passed to a classification head.

In [ ]:
class SRMConv(nn.Module):
    """
    Fixed convolutional layer applying three Spatial Rich Model (SRM) filters.

    These filters extract noise residuals from images forensic artifacts that
    reveal which generative model produced the image. Weights are fixed (not trained).
    """
    def __init__(self):
        super().__init__()

        # high-pass filter sensitive to local pixel inconsistencies
        f1 = np.array([[ 0,  0,  0,  0,  0],
                       [ 0, -1,  2, -1,  0],
                       [ 0,  2, -4,  2,  0],
                       [ 0, -1,  2, -1,  0],
                       [ 0,  0,  0,  0,  0]]) / 4.0

        # captures second-order noise statistics across a wider neighborhood
        f2 = np.array([[-1,  2, -2,  2, -1],
                       [ 2, -6,  8, -6,  2],
                       [-2,  8,-12,  8, -2],
                       [ 2, -6,  8, -6,  2],
                       [-1,  2, -2,  2, -1]]) / 12.0

        # simple edge-residual filter
        f3 = np.array([[ 0,  0,  0,  0,  0],
                       [ 0,  0,  0,  0,  0],
                       [ 0,  1, -2,  1,  0],
                       [ 0,  0,  0,  0,  0],
                       [ 0,  0,  0,  0,  0]]) / 2.0

        filters = np.stack([f1, f2, f3], axis=0)       # (3, 5, 5)
        filters = np.expand_dims(filters, axis=1)       # (3, 1, 5, 5)
        # apply each filter to all 3 RGB channels independently
        filters = np.repeat(filters, 3, axis=1)         # (3, 3, 5, 5)

        # requires_grad=False keeps these weights frozen during training
        self.weight = nn.Parameter(
            torch.tensor(filters, dtype=torch.float32),
            requires_grad=False
        )

    def forward(self, x):
        return nn.functional.conv2d(x, self.weight, padding=2)

In [ ]:
class DualStreamModel(nn.Module):
    """
    Two-stream classification model for synthetic image attribution.

    Stream 1 processes the raw RGB image through EfficientNet-B4.
    Stream 2 applies SRM noise filters first, then passes the result through
    a second EfficientNet-B4. Both feature vectors are concatenated and fed
    into a fully connected classification head.

    Args:
        num_classes: number of output classes (10 for this challenge)
    """
    def __init__(self, num_classes=10):
        super().__init__()

        # stream 1: standard RGB features
        self.rgb_backbone = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
        rgb_features = self.rgb_backbone.classifier[1].in_features
        # replace classifier with identity to get raw feature vectors
        self.rgb_backbone.classifier = nn.Identity()
        self.backbone_name = self.rgb_backbone.__class__.__name__

        # stream 2: forensic noise features via SRM
        self.srm = SRMConv()
        self.srm_backbone = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
        self.srm_backbone.classifier = nn.Identity()

        # fusion head: takes concatenated features from both streams
        self.classifier = nn.Sequential(
            nn.Linear(rgb_features * 2, 512),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        rgb_feat = self.rgb_backbone(x)
        srm_feat = self.srm_backbone(self.srm(x))
        combined = torch.cat([rgb_feat, srm_feat], dim=1)
        return self.classifier(combined)

## Training Setup

We instantiate the model and configure the optimizer, loss function, learning rate scheduler, and mixed precision scaler.

We use AdamW with OneCycleLR the scheduler ramps the learning rate up then down over the course of training, which often converges faster than a fixed LR. Mixed precision (float16) cuts memory usage and speeds up training on modern GPUs.

In [ ]:
model = DualStreamModel(num_classes=NUM_OF_CLASSES).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)

# wrap in DataParallel if multiple GPUs are available
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    # preserve backbone_name attribute through the DataParallel wrapper
    model.backbone_name = model.module.backbone_name

criterion = nn.CrossEntropyLoss()

# OneCycleLR steps every batch, not every epoch
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=len(train_dl),
    epochs=EPOCHS
)

# GradScaler enables mixed precision training
scaler = torch.amp.GradScaler(device)

## Training Loop

The `fit` function runs the full training and validation loop. It saves the best checkpoint whenever validation accuracy improves, and stops early if there's no improvement for `PATIENCE` epochs.

In [ ]:
def fit(model):
    """
    Train the model with early stopping based on validation accuracy.

    Each epoch runs the training loop, evaluates on validation, saves the
    best checkpoint, and stops if val accuracy doesn't improve for PATIENCE epochs.

    Args:
        model: nn.Module to train (uses global optimizer, criterion, scheduler, scaler)

    Returns:
        history: list of dicts with train_loss, train_acc, val_loss, val_acc per epoch
    """
    history  = []
    best_acc = 0.0
    counter  = 0

    for epoch in range(EPOCHS):
        train_all_loss = []
        train_all_acc  = []

        model.train()

        for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs, labels = batch
            imgs   = imgs.to(device)
            labels = labels.to(device)

            # zero gradients before the forward pass
            optimizer.zero_grad()

            # autocast runs the forward pass in float16 for speed
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(imgs)
                loss    = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            # OneCycleLR must step every batch
            scheduler.step()

            _, preds = torch.max(outputs, dim=1)
            train_all_loss.append(loss.detach().cpu().item())
            train_all_acc.append((preds == labels).float().mean().item())

        train_loss = sum(train_all_loss) / len(train_all_loss)
        train_acc  = sum(train_all_acc)  / len(train_all_acc)

        val_all_loss = []
        val_all_acc  = []

        model.eval()
        with torch.no_grad():
            for batch in val_dl:
                imgs, labels = batch
                imgs   = imgs.to(device)
                labels = labels.to(device)

                outputs = model(imgs)
                loss    = criterion(outputs, labels)

                _, preds = torch.max(outputs, dim=1)
                val_all_loss.append(loss.detach().cpu().item())
                val_all_acc.append((preds == labels).float().mean().item())

        val_loss = sum(val_all_loss) / len(val_all_loss)
        val_acc  = sum(val_all_acc)  / len(val_all_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        history.append({
            "train_loss": train_loss, "train_acc": train_acc,
            "val_loss":   val_loss,   "val_acc":   val_acc
        })

        if val_acc > best_acc:
            best_acc = val_acc
            counter  = 0
            torch.save(model.state_dict(), f"best_{model.backbone_name}_model.pth")
            print("  >> best model saved")
        else:
            counter += 1
            print(f"  No improvement ({counter}/{PATIENCE})")
            if counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    return history

## Inference with Test-Time Augmentation

Instead of predicting once per image, we run each test image through the model `n_augments` times with slightly different augmentations and average the softmax probabilities. This reduces variance and usually improves accuracy by a small margin.

In [ ]:
def predict_with_tta(model, loader, device, n_augments=5):
    """
    Run inference with Test-Time Augmentation (TTA).

    Each image is augmented n_augments times and the softmax probabilities
    are averaged before taking the argmax prediction.

    Args:
        model: trained nn.Module in eval mode
        loader: DataLoader that returns (imgs, ids) pairs
        device: torch device to run inference on
        n_augments: number of augmented views to average per image

    Returns:
        all_ids: list of image IDs matching the test CSV
        all_preds: list of predicted class labels (integers 0-9)
    """
    model.eval()

    # light augmentation enough to add diversity without destroying forensic features
    tta_transform = v2.Compose([
        v2.Resize((SIZE, SIZE)),
        v2.RandomHorizontalFlip(p=0.5),
        v2.ColorJitter(brightness=0.1, contrast=0.1),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(MEAN_NORM, STD_NORM)
    ])

    all_ids, all_preds = [], []

    with torch.no_grad():
        for imgs, ids in tqdm(loader):
            imgs = imgs.to(device)

            # accumulate softmax probabilities across all augmented views
            probs = torch.zeros(imgs.size(0), NUM_OF_CLASSES).to(device)
            for _ in range(n_augments):
                aug_imgs = torch.stack([tta_transform(img) for img in imgs])
                aug_imgs = aug_imgs.to(device)
                outputs  = model(aug_imgs)
                probs   += torch.softmax(outputs, dim=1)

            preds = probs.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_ids.extend(ids.tolist() if torch.is_tensor(ids) else ids)

    return all_ids, all_preds

## Run Training

Let's train the model. The best checkpoint is saved automatically.

In [ ]:
history = fit(model)

## Generate Submission

We load the best checkpoint, run inference with TTA on the test set, and save the predictions as `submission.csv`.

In [ ]:
# load the best checkpoint saved during training
model.load_state_dict(torch.load(f"best_{model.backbone_name}_model.pth", map_location=device))

all_ids, all_preds = predict_with_tta(model, test_dl, device, n_augments=5)

submission = pd.DataFrame({"ID": all_ids, "TARGET": all_preds})
submission = submission.sort_values("ID").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)

print(submission.head(10))
print(f"Total predictions: {len(submission)}")